# LightLogger Data Processing Pipeline

## Overview

This notebook walks through the preprocessing stages implemented in `preprocessing_pipeline.py` for the LightLogger / FLIC dataset. The goal is to take raw world-camera and Neon eye-tracking recordings and turn them into validated, analysis-ready outputs for virtual foveation, SPD generation, and downstream summaries.

The workflow is intentionally staged. Early steps focus on transfer, download, and integrity checks; middle steps build synchronized world and Neon representations; later steps derive virtually foveated videos and SPDs. Running the notebook in order helps ensure that each stage receives the files and directory structure expected by the next one.

## Pipeline Stages

1. Acquire and verify raw data
2. Generate stitched world-camera videos from `GKA` chunk recordings
3. Verify Neon and world recordings are correctly paired
4. Run the egocentric mapper on Neon recordings and world videos
5. Clean eye-tracking data and save the raw/cleaned comparison PDF
6. Run the world-camera transformation pipeline
7. Tag task start/end windows and generate virtually foveated videos
8. Generate SPDs and aggregate summary outputs

Throughout the notebook, the markdown mirrors the responsibilities of the corresponding functions in `preprocessing_pipeline.py` so the processing logic and the documentation stay aligned.



## Environment Setup

Import the pipeline module and load the profiling extensions used for long-running preprocessing steps. Most cells in this notebook reload `preprocessing_pipeline` before execution so local code changes are picked up without restarting the kernel.



In [1]:
import importlib
import preprocessing_pipeline

In [2]:
importlib.reload(preprocessing_pipeline)
print(preprocessing_pipeline.get_activity_names())
print(preprocessing_pipeline.get_subject_ids())

{'chat', 'sitBiopond', 'walkBiopond', 'walkIndoor', 'read', 'dark', 'walkOutdoor'}
{1038: 32, 28: 20, 42: 29, 51: 29, 1047: 30, 1044: 39, 21: 18, 20: 18, 1052: 24, 1051: 33, 1054: 31, 54: 20, 52: 32, 1057: 27, 1056: 34, 1058: 27, 55: 32, 1062: 33, 1055: 33, 1059: 31, 1063: 30, 1066: 27, 56: 30, 58: 40, 1067: 25, 60: 34, 57: 39, 1068: 22, 61: 30, 62: 33}


In [ ]:
#need to do 1044 here
preprocessing_pipeline.populate_FLIC_data(overwrite_existing=False, verbose=True, subjects_to_process=[21, 42, 51, 52, 54, 1038, 1044, 1054], activities_to_process=preprocessing_pipeline.get_activity_names())

In [ ]:
len(preprocessing_pipeline.get_subject_ids().keys() - {2001, 2002, 2003, 2004, 2005, 2006})

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_data_integrity(verbose=True, output_path="/Users/zacharykelly/Desktop/testingDataCuration.xlsx", subjects_and_activities_to_process= {subject_id: preprocessing_pipeline.get_activity_names() for subject_id in preprocessing_pipeline.get_subject_ids()})# {subject: preprocessing_pipeline.get_activity_names() for subject in preprocessing_pipeline.get_subject_ids()})# {55: set(["sitBiopond"]), 1062: set(["walkBiopond"]), 1063: set(["walkBiopond"]), 1066: set(["walkBiopond"]), 1067: set(["dark"]), 62: set(["chat", "dark", "read", "sitBiopond"])}
                                                                                     


Processing Subjects:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Processing Recordings:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Recordings:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Recordings:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Sensors:   0%|          | 0/2 [00:00<?, ?it/s]

... (42 outputs left out)

Processing Chunks:   0%|          | 0/25 [00:00<?, ?it/s]

Processing Chunks:   0%|          | 0/11 [00:00<?, ?it/s]

Processing Recordings:   0%|          | 0/1 [00:00<?, ?it/s]

{55: {'meta_problems': [],
  'activities': {'chat': {'meta_problems': [], 'GKA': [], 'Neon': []},
   'dark': {'meta_problems': [], 'GKA': [], 'Neon': []},
   'read': {'meta_problems': [], 'GKA': [], 'Neon': []},
   'sitBiopond': {'meta_problems': [], 'GKA': [], 'Neon': []},
   'walkBiopond': {'meta_problems': [], 'GKA': [], 'Neon': []},
   'walkIndoor': {'meta_problems': [], 'GKA': [], 'Neon': []},
   'walkOutdoor': {'meta_problems': [], 'GKA': [], 'Neon': []}}}}

In [ ]:
raw_data_integrity_issues.keys()

# Step 0 - Acquire and Verify Raw Recordings

## Purpose

Prepare the raw dataset so every requested subject/activity has the source material expected by downstream functions. In `preprocessing_pipeline.py`, this stage is handled by the transfer/download helpers plus the integrity checks that validate the required `GKA` and `Neon` directory contents.

## What We Check Here

- Light Logger world recordings are present in each activity's `GKA` folder
- Neon scene-video exports and time-series files are present in each activity's `Neon` folder
- Required Neon metadata and CSV tables exist before mapping or foveation begins
- The raw directory structure is consistent across the requested subjects and activities

## Expected Raw Layout

```text
FLIC_XXXX/
    activity_name/
        GKA/
        Neon/
```

Catching missing folders or incomplete downloads here is important because later stages assume these raw inputs already exist and are internally consistent.



### Transfer Light Logger World-Camera Recordings

Use `transfer_light_logger_recordings(...)` to copy the raw Light Logger recordings into the dataset root. This step is responsible for bringing the world-camera source material into the expected subject/activity structure before any video generation begins.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.transfer_light_logger_recordings(src_dir="/Volumes/EXTERNAL_1",
                                                        overwrite_existing=False, 
                                                        verbose=True,
                                                        subjects_and_activities_to_process={62: preprocessing_pipeline.get_activity_names()},
                                                        )

### Download Pupil Labs Neon Exports

Use the Pupil Cloud download helper to retrieve the Neon `Timeseries Data + Scene Video` package for the same recordings. These exports provide the scene video and the gaze-related CSV files required by the later integrity, pairing, and egocentric-mapping steps.



#### Download the Neon `Timeseries Data + Scene Video` Packages

`download_pupil_cloud_recordings(...)` downloads the raw Neon export archives into the raw dataset directory. Keep this step early in the workflow so the subsequent verification cells can confirm that the expected `Neon` contents are present before any downstream processing starts.



In [ ]:


importlib.reload(preprocessing_pipeline)
api_key: str = preprocessing_pipeline.get_pupil_cloud_apikey()
preprocessing_pipeline.download_pupil_cloud_recordings(api_key, 
                                                       dst_dir="/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/usableSubjects", 
                                                       verbose=True,
                                                       overwrite_existing=False,
                                                       subjects_and_activities_to_process={62: preprocessing_pipeline.get_activity_names()}
                                                      )




### Verify Raw `GKA` and `Neon` Contents

Run the integrity checks immediately after transfer/download. `verify_data_integrity(...)` checks that both modalities exist for each requested recording, while `verify_neon_integrity(...)` performs a more detailed pass over the extracted Neon contents such as `gaze.csv`, `fixations.csv`, `world_timestamps.csv`, and the scene video file.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_neon_integrity(verbose=True)

# Step 1 - Generate World-Camera Videos

## Purpose

Convert raw Light Logger chunk recordings stored in each activity's `GKA` folder into a single playable `W.avi` file per recording. This section corresponds to `generate_world_videos(...)` in `preprocessing_pipeline.py`.

## What This Step Does

- Finds every requested subject/activity with a valid raw `GKA` folder
- Reads the world-camera chunk files for that recording
- Optionally applies the preprocessing switches used by the pipeline, including color weighting, floor/ceiling correction, fielding correction, dark-noise removal, debayering, digital gain, and missing-frame filling
- Writes the stitched world video to the processing directory as `GKA/W.avi`

This world video is a required input for both the Neon/world pairing check and the egocentric mapper.



### Build `W.avi` Files from Raw `GKA` Chunks

Execute `generate_world_videos(...)` here after the raw data has been verified. Successful completion of this block means each selected recording now has a processed world-camera video available for synchronization and mapping.



In [ ]:



importlib.reload(preprocessing_pipeline)

# First, let's get the totally fine subjects 
totally_fine: dict = {subject: preprocessing_pipeline.get_activity_names() for subject in [21, 42, 51, 52, 54, 56, 57, 58, 60, 61, 1038, 1044, 1055, 1059, 1068]}

manually_checked_that_are_fine: dict = {
    20: {"chat", "dark", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    21: {"chat", "dark", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    28: {"chat", "dark", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    55: {"chat", "read", "walkIndoor", "walkOutdoor"},
    1047: {"chat", "read", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    1051: {"chat", "dark", "read", "walkBiopond", "walkIndoor", "walkOutdoor"},
    1052: {"chat", "read", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    1054: {"chat", "read", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    1056: {"dark", "read", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
    1057: {"read", "sitBiopond", "walkIndoor", "walkOutdoor"},
    1058: {"chat", "dark", "read", "sitBiopond", "walkIndoor", "walkOutdoor"},
    1062: {"chat", "dark", "read", "sitBiopond", "walkIndoor", "walkOutdoor"},
    1063: {"dark", "read", "sitBiopond", "walkIndoor", "walkOutdoor"},
    1066: {"chat", "dark", "read", "walkIndoor", "walkOutdoor"},
    1067: {"chat", "read", "sitBiopond", "walkBiopond", "walkIndoor", "walkOutdoor"},
}


subjects_and_activities_to_process = totally_fine | manually_checked_that_are_fine


preprocessing_pipeline.generate_world_videos(
    overwrite_existing=True,
    verbose=True,
    apply_digital_gain=True,
    apply_color_weights=False,
    debayer_images=False,
    apply_floor_ceiling=False,
    apply_fielding_function=False,
    linearize_camera_responsivity=False,
    subjects_and_activities_to_process={62: set(["sitBiopond"])}
)



# Step 2 - Verify Neon and World Pairing

## Purpose

Confirm that each processed world video corresponds to the correct raw Neon recording before we run the egocentric mapper. In `preprocessing_pipeline.py`, `verify_world_neon_pairing(...)` loads the two modalities for the same subject/activity and helps confirm they belong to the same session.

## Why This Matters

A mismatched world/Neon pair can silently contaminate all downstream outputs. The pairing check is therefore one of the highest-value validation steps in the notebook. With `verbose=True`, the function displays frames that make it easier to visually confirm that the recordings match.



### Review World-Video and Neon Pairing

Run `verify_world_neon_pairing(...)` after world-video generation and before egocentric mapping. Treat any mismatch here as a blocker for the later synchronization-dependent steps.



In [ ]:
importlib.reload(preprocessing_pipeline)

# TODO: MAYBE I CAN GET RID OF THIS

# Return value is a dictionary of the recordings whose integrity was verified
videos_observed: dict[str, dict[str, bool]] = preprocessing_pipeline.verify_world_neon_pairing(verbose=True)

# Step 3 - Run the Egocentric Video Mapper

## Purpose

Generate the Neon-to-world alignment products used for gaze and fixation mapping. This section corresponds to `generate_egocentric_mapper_results(...)`, which uses the Neon recording directory plus the processed `W.avi` world video for each activity.

## Notes

- This step depends on the raw Neon export and the processed world video both being present
- The mapper can produce gaze mappings, fixation mappings, or both
- The notebook note about switching to the `egocentric_video_mapper` kernel still applies for this stage



In [ ]:
importlib.reload(preprocessing_pipeline)

# [21, 42, 51, 52, 54, 1038, 1044, 1054, 1055, 1059]

preprocessing_pipeline.generate_egocentric_mapper_results(overwrite_existing=True, verbose=True, subjects_to_process=[1044], activities_to_process= preprocessing_pipeline.get_activity_names())#preprocessing_pipeline.get_activity_names()) 

In [ ]:
import importlib
import preprocessing_pipeline

importlib.reload(preprocessing_pipeline)

# Subjects with no issues in the integrity-check output above.
subjects_to_process = [20, 57, 60, 61]
activities_to_process = preprocessing_pipeline.get_activity_names()

preprocessing_pipeline.generate_egocentric_mapper_results(
    overwrite_existing=True,
    verbose=True,
    subjects_to_process=subjects_to_process,
    activities_to_process=activities_to_process,
)


In [ ]:
preprocessing_pipeline.clean_eye_data(
    overwrite_existing=False,
    verbose=True,
    subjects_to_process=[57, 60, 61],
    activities_to_process=preprocessing_pipeline.get_activity_names(),
)


# Step 4 - Clean Eye-Tracking Data

Run after egocentric mapping so each Neon attempt has `alternative_camera_gaze.csv` available. This stage cleans pupil diameter, eyelid aperture, and mapped gaze and writes `cleaned_eye_data.csv` in `Neon/<attempt>/egocentric_mapper_results/`.

`verbose=True` is the default: display Geoff's six-panel raw-versus-cleaned plot and save it as vector graphics in `eye_cleaned_vs_raw.pdf` beside the CSV. Set `verbose=False` to disable progress and plots. Existing CSVs are skipped unless `overwrite_existing=True`; use that option to regenerate existing outputs with the PDF.


In [ ]:
importlib.reload(preprocessing_pipeline)
importlib.reload(preprocessing_pipeline.cleanEyeTracking)


preprocessing_pipeline.clean_eye_data(
    overwrite_existing=False,
    verbose=True,
    subjects_to_process=[1059],
    activities_to_process=["walkOutdoor"]
)


# Step 5 - Run Transformation Pipeline

## Purpose

Run `run_transformation_pipeline(...)` on the selected recordings. It calls `sensors_utility.process_raw_recording(...)` for each numbered GKA recording, processing world data first and then minispectrometer (MS) data.

Foveated opponent-channel radiance is saved to `GKA/<recording_number>/W/transformed.hdf5`. The file contains `/data/frames`, `/agcSettings`, `/metadata`, and `/data/average_frames`. World maps are stored as float64 in (frame, azimuth, elevation, channel) order, preserving signed fractional values and NaNs. `/data/cone_weights` stores the effective LMS weights. RGB averages are taken before opponent conversion and retain floating-point precision. The averages are spatial RGB means for each frame, aligned with `/agcSettings/timestamps`; they are computed while each transformed chunk is in memory.

MS processing reads these small RGB arrays, averages them over midpoint windows around the MS timestamps (including windows spanning chunks), and fits joint MS/camera spectra. Results are saved to `GKA/<recording_number>/M/transformed.hdf5`, with compressed, chunked arrays under `/data` (radiance, `S`, `fVal`, and fit errors), `/timestamps` in seconds, and `/metadata` for timing diagnostics and fit settings.

With `overwrite_existing=False`, completed world files and completed MS files are skipped independently. The joint MS stage requires saved RGB averages. Set `overwrite_existing=True` to regenerate older world files without them; opponent-channel maps cannot substitute for RGB averages.

Set both `n_workers=1` and `demosaic_n_workers=1` for fully serial processing.


In [ ]:
import importlib
import preprocessing_pipeline

importlib.reload(preprocessing_pipeline)
importlib.reload(preprocessing_pipeline.sensors_utility)

preprocessing_pipeline.run_transformation_pipeline(
    overwrite_existing=True,
    verbose=True,
    subjects_to_process=[20],
    activities_to_process=["walkOutdoor"],
)

# Step 6 - Generate Virtually Foveated Videos

## Purpose

Prepare the inputs needed for retinally centered video generation and then run the virtual foveation stage implemented by `generate_tag_task_start_ends(...)` and `generate_virtually_foveated_videos(...)`.

## What Happens in This Stage

1. Identify the task window for each recording
2. Use the tagged time bounds together with the mapped Neon outputs to drive virtual foveation
3. Optionally verify that the expected foveated outputs were produced

These outputs feed directly into the SPD-generation stage that follows later in the notebook.



### Tag Task Start and End Windows

Run `generate_tag_task_start_ends(...)` to locate the usable portion of each recording before virtual foveation begins. These tags define the time window that the MATLAB-based foveation step will use downstream.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_tag_task_start_ends(overwrite_existing=False, verbose=True)

### Build the Virtually Foveated Videos

Execute `generate_virtually_foveated_videos(...)` once the task windows have been tagged. After this step, the processing directory should contain the retinally centered video products that the SPD routines consume later in the pipeline.



In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_virtually_foveated_videos(overwrite_existing=True, verbose=True,  
                                                          activities_to_process=["chat", "walkBiopond"])

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_virtually_foveated_video_integrity(verbose=True, activities_to_skip=["phone", "lunch"])

## 7. Generate SPDs

### 1. Generate the Indinvidual SPDs and plots for all desired subjects and activitys 

In [ ]:
importlib.reload(preprocessing_pipeline)

color_modes: list[str] = ["a", "c_lm", "c_s"]
for color_mode in color_modes:
    preprocessing_pipeline.generate_spds(color_mode=color_mode,
                                        overwrite_existing=True, 
                                        verbose=True, 
                                        activities_to_process=["chat", "walkBiopond"], 
                                        )

In [ ]:
# Let's combine the SPDs for the desired colormodes 
importlib.reload(preprocessing_pipeline)

preprocessing_pipeline.combine_spds(overwrite_existing=True, 
                                    verbose=True, 
                                    activities_to_process=["chat", "walkBiopond"],
                                    color_modes_to_process=["a", "c_lm", "c_s"]
                                  )


### 2. Plot the indivudal SPDs (optionally if not done above)
We also have this as a separate optional step because the first step is ridiculously long in terms of runtime

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.adjust_spd_axes(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True)

#### 3. For each subject, output a grouped plot of their results for all activities


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_per_subject(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 4. Generate average SPDs for each activity by averaging across subjects for that activity

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_subject(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                    combine_figures=True, common_axes=True)

#### 5. Group the mean SPDs for each activity by the activity type


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_across_subjects(overwrite_existing=True, color_mode="L-M", verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 6. Generate average SPDs across all activities' averages

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_all(overwrite_existing=True, verbose=True, color_mode="L-M", activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True, common_axes=True)

In [ ]:
importlib.reload(preprocessing_pipeline)

sitting_vs_walking: dict = {"sitting": ["sitBiopond", "work", "chat"], 
                            "walking": ["walkBiopond", "walkIndoor", "walkOutdoor"]
                           }

indoor_vs_outdoor: dict = {"indoor": ["work", "chat", "walkIndoor"], 
                           "outdoor": ["walkOutdoor", "sitBiopond", "walkBiopond"]
                          }

preprocessing_pipeline.generate_spds_across_groups(overwrite_existing=True, 
                                                   color_mode="L-M",
                                                   verbose=True, 
                                                   activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                   combine_figures=True, 
                                                   common_axes=True,
                                                   groups=sitting_vs_walking
                                                )

### Generate actigraphy graphs

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_actigraphy_graphs(overwrite_exsiting=True, 
                                                  verbose=True, 
                                                  subjects_to_process=[2001],
                                                  activities_to_process=["walkBiopond"]
                                                )

In [ ]:
import matlab.engine
eng = matlab.engine.start_matlab()

print(eng.version(nargout=1))